# MAE（Masked Autoencoder）on CIFAR-10

这个 Notebook 从零实现 `MAE（Masked Autoencoder）`，在 CIFAR-10 上完成自监督预训练，再 fine-tune 做分类。

内容包括：
- CIFAR-10 数据加载与 Patch 化
- MAE Encoder（只处理可见 patch）与 Decoder（重建所有 patch）
- 随机掩码策略（75% 遮盖率）
- 预训练（重建损失）流程
- 重建结果可视化（原图 / 掩码图 / 重建图三联）
- Fine-tune 分类
- 与 MoCo / SimCLR（对比式）的范式对比

## 1. 环境准备

```bash
pip install torch torchvision matplotlib
```

In [ ]:
import math
from dataclasses import dataclass

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

plt.style.use('seaborn-v0_8')
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
@dataclass
class Config:
    data_root: str = './data'
    image_size: int = 32
    # CIFAR-10 图小，用 4x4 patch
    patch_size: int = 4
    batch_size: int = 256
    num_workers: int = 2
    embed_dim: int = 192
    encoder_depth: int = 6
    decoder_dim: int = 96
    decoder_depth: int = 2
    num_heads: int = 6
    mlp_ratio: float = 4.0
    # 掩码率 75%，论文发现比 50% 或 90% 更优
    mask_ratio: float = 0.75
    lr_pretrain: float = 1.5e-4
    pretrain_epochs: int = 30
    lr_finetune: float = 1e-3
    finetune_epochs: int = 10
    num_classes: int = 10

cfg = Config()
num_patches = (cfg.image_size // cfg.patch_size) ** 2
print(f'每张图切成 {num_patches} 个 patch（{cfg.image_size//cfg.patch_size}×{cfg.image_size//cfg.patch_size}）')

## 2. 加载 CIFAR-10

In [ ]:
cifar10_mean = (0.4914, 0.4822, 0.4465)
cifar10_std  = (0.2470, 0.2435, 0.2616)

pretrain_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

finetune_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

pretrain_dataset = datasets.CIFAR10(cfg.data_root, train=True, download=True, transform=pretrain_transform)
finetune_dataset = datasets.CIFAR10(cfg.data_root, train=True, download=False, transform=finetune_transform)
val_dataset      = datasets.CIFAR10(cfg.data_root, train=False, download=False, transform=val_transform)

## 3. 构建 DataLoader

In [ ]:
pretrain_loader = DataLoader(pretrain_dataset, batch_size=cfg.batch_size, shuffle=True,
                             num_workers=cfg.num_workers, pin_memory=torch.cuda.is_available())
finetune_loader = DataLoader(finetune_dataset, batch_size=cfg.batch_size, shuffle=True,
                             num_workers=cfg.num_workers, pin_memory=torch.cuda.is_available())
val_loader      = DataLoader(val_dataset,      batch_size=cfg.batch_size, shuffle=False,
                             num_workers=cfg.num_workers, pin_memory=torch.cuda.is_available())
print('DataLoader 构建完成')

## 4. MAE 实现

### 4.1 核心思想：视觉版 BERT

| | BERT | MAE |
|-|------|-----|
| 输入 | 文本 token | 图像 patch |
| 掩码对象 | 随机 token（15%） | 随机 patch（75%） |
| 重建目标 | 被掩盖的词 | 被掩盖 patch 的像素值 |
| Encoder | 处理全部 token | **只处理可见 patch**（高效） |

### 4.2 与对比式自监督的区别

- **MoCo / SimCLR / BYOL**：对比式，需要数据增强构造正负样本对，依赖增强策略设计
- **MAE**：生成式，通过重建被遮住的 patch 学习表示，不依赖负样本

In [ ]:
class PatchEmbed(nn.Module):
    def __init__(self, image_size, patch_size, in_channels=3, embed_dim=192):
        super().__init__()
        self.num_patches = (image_size // patch_size) ** 2
        self.patch_size = patch_size
        # 用步长等于 patch_size 的卷积实现无重叠切分
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)          # B, embed_dim, H/P, W/P
        x = x.flatten(2)         # B, embed_dim, num_patches
        x = x.transpose(1, 2)    # B, num_patches, embed_dim
        return x


class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        mlp_dim = int(dim * mlp_ratio)
        self.mlp  = nn.Sequential(
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, dim),
        )

    def forward(self, x):
        # Pre-LN 更稳定，Post-LN 原始 Transformer 的做法容易梯度爆炸
        h = self.norm1(x)
        x = x + self.attn(h, h, h)[0]
        x = x + self.mlp(self.norm2(x))
        return x


class MAE(nn.Module):
    def __init__(self, cfg, num_patches):
        super().__init__()
        self.patch_embed = PatchEmbed(cfg.image_size, cfg.patch_size, embed_dim=cfg.embed_dim)
        self.num_patches = num_patches
        self.mask_ratio  = cfg.mask_ratio
        self.patch_size  = cfg.patch_size

        # 可见 patch 的位置编码
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, cfg.embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.encoder = nn.Sequential(*[
            TransformerBlock(cfg.embed_dim, cfg.num_heads, cfg.mlp_ratio)
            for _ in range(cfg.encoder_depth)
        ])
        self.encoder_norm = nn.LayerNorm(cfg.embed_dim)

        # 投影层把 encoder 维度映射到 decoder 维度
        self.decoder_embed = nn.Linear(cfg.embed_dim, cfg.decoder_dim)
        # mask token：被遮住的 patch 用这个可学习向量占位
        self.mask_token    = nn.Parameter(torch.zeros(1, 1, cfg.decoder_dim))
        self.decoder_pos   = nn.Parameter(torch.zeros(1, num_patches, cfg.decoder_dim))
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        nn.init.trunc_normal_(self.decoder_pos, std=0.02)

        self.decoder = nn.Sequential(*[
            TransformerBlock(cfg.decoder_dim, max(1, cfg.num_heads // 2))
            for _ in range(cfg.decoder_depth)
        ])
        self.decoder_norm = nn.LayerNorm(cfg.decoder_dim)
        # 每个 patch 还原为像素值：patch_size^2 * 3
        self.decoder_pred = nn.Linear(cfg.decoder_dim, cfg.patch_size ** 2 * 3)

    def random_masking(self, x):
        B, N, D = x.shape
        num_keep = int(N * (1 - self.mask_ratio))

        # 随机打乱顺序，取前 num_keep 个作为可见 patch
        noise = torch.rand(B, N, device=x.device)
        ids_shuffle = noise.argsort(dim=1)
        ids_restore = ids_shuffle.argsort(dim=1)

        ids_keep = ids_shuffle[:, :num_keep]
        x_visible = torch.gather(x, 1, ids_keep.unsqueeze(-1).expand(-1, -1, D))

        mask = torch.ones(B, N, device=x.device)
        mask[:, :num_keep] = 0
        mask = torch.gather(mask, 1, ids_restore)

        return x_visible, mask, ids_restore

    def forward(self, x):
        tokens = self.patch_embed(x) + self.pos_embed
        tokens_vis, mask, ids_restore = self.random_masking(tokens)

        # Encoder 只看可见 patch，计算量远小于全部 patch
        encoded = self.encoder(tokens_vis)
        encoded = self.encoder_norm(encoded)

        dec_tokens = self.decoder_embed(encoded)
        B, num_vis, _ = dec_tokens.shape
        num_mask = self.num_patches - num_vis

        # 将 mask token 插入被掩盖位置，恢复完整序列供 Decoder 处理
        mask_tokens = self.mask_token.expand(B, num_mask, -1)
        full = torch.cat([dec_tokens, mask_tokens], dim=1)
        full = torch.gather(full, 1, ids_restore.unsqueeze(-1).expand(-1, -1, full.size(-1)))
        full = full + self.decoder_pos

        decoded = self.decoder(full)
        decoded = self.decoder_norm(decoded)
        pred = self.decoder_pred(decoded)  # B, N, patch_size^2*3

        return pred, mask

    def encode(self, x):
        # Fine-tune 时只用 Encoder，不掩码
        tokens = self.patch_embed(x) + self.pos_embed
        encoded = self.encoder(tokens)
        encoded = self.encoder_norm(encoded)
        return encoded.mean(dim=1)  # 全局平均池化作为图像表示


mae_model = MAE(cfg, num_patches).to(device)
print('MAE 构建完成')

## 5. 掩码策略解读

In [ ]:
@torch.no_grad()
def visualize_masking(model, images, cfg, device):
    model.eval()
    x = images[:4].to(device)
    tokens = model.patch_embed(x) + model.pos_embed
    _, mask, _ = model.random_masking(tokens)

    # mask 形状 B, N，1 表示被遮住
    H = W = cfg.image_size // cfg.patch_size
    mask_2d = mask.view(-1, H, W).cpu()

    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    mean = torch.tensor(cifar10_mean).view(3, 1, 1)
    std  = torch.tensor(cifar10_std).view(3, 1, 1)

    for i in range(4):
        img = (images[i] * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
        axes[0, i].imshow(img)
        axes[0, i].set_title('原图')
        axes[0, i].axis('off')

        # 把掩码放大回图像尺寸展示
        mask_img = mask_2d[i].unsqueeze(0).repeat(cfg.patch_size, 1, cfg.patch_size, 1)
        # 简单用灰色覆盖被掩码区域
        masked = img.copy()
        m = mask_2d[i].numpy()
        for r in range(H):
            for c in range(W):
                if m[r, c] == 1:
                    masked[r*cfg.patch_size:(r+1)*cfg.patch_size,
                           c*cfg.patch_size:(c+1)*cfg.patch_size] = 0.5
        axes[1, i].imshow(masked)
        axes[1, i].set_title(f'掩码（遮盖 {cfg.mask_ratio*100:.0f}%）')
        axes[1, i].axis('off')

    plt.tight_layout()
    plt.show()


sample_imgs, _ = next(iter(pretrain_loader))
cifar10_mean_t = (0.4914, 0.4822, 0.4465)
cifar10_std_t  = (0.2470, 0.2435, 0.2616)
visualize_masking(mae_model, sample_imgs, cfg, device)

## 6. 参数量统计

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'MAE 总参数量: {count_parameters(mae_model):,}')

## 7. 预训练函数（重建损失）

In [ ]:
def patchify(imgs, patch_size):
    # 将图像转成 patch 序列，用于与 Decoder 预测值计算 MSE
    B, C, H, W = imgs.shape
    p = patch_size
    h = H // p
    w = W // p
    imgs = imgs.reshape(B, C, h, p, w, p)
    imgs = imgs.permute(0, 2, 4, 3, 5, 1)  # B, h, w, p, p, C
    imgs = imgs.reshape(B, h * w, p * p * C)
    return imgs


pretrain_optimizer = optim.AdamW(mae_model.parameters(), lr=cfg.lr_pretrain, weight_decay=0.05)


def pretrain_one_epoch(model, dataloader, optimizer, device, patch_size):
    model.train()
    total_loss = 0.0
    for images, _ in dataloader:
        images = images.to(device)
        pred, mask = model(images)

        target = patchify(images, patch_size)
        # 只在被掩盖的 patch 上计算损失（可见 patch 已知，无需重建）
        loss = ((pred - target) ** 2).mean(dim=-1)
        loss = (loss * mask).sum() / mask.sum()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(dataloader)

## 8. 预训练主循环

In [ ]:
pretrain_losses = []

for epoch in range(cfg.pretrain_epochs):
    loss = pretrain_one_epoch(mae_model, pretrain_loader, pretrain_optimizer, device, cfg.patch_size)
    pretrain_losses.append(loss)
    print(f'Pretrain Epoch {epoch+1:3d}/{cfg.pretrain_epochs}  loss={loss:.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(pretrain_losses)
ax.set_title('MAE 预训练重建损失')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
plt.tight_layout()
plt.show()

## 9. 重建结果可视化

In [ ]:
@torch.no_grad()
def show_reconstruction(model, images, cfg, device):
    model.eval()
    x = images[:4].to(device)
    pred, mask = model(x)

    # 将预测的 patch 序列重组回图像
    p = cfg.patch_size
    H = W = cfg.image_size // p
    pred_img = pred.cpu().reshape(4, H, W, p, p, 3).permute(0, 5, 1, 3, 2, 4).reshape(4, 3, cfg.image_size, cfg.image_size)

    mean = torch.tensor(cifar10_mean).view(1, 3, 1, 1)
    std  = torch.tensor(cifar10_std).view(1, 3, 1, 1)

    orig   = (images[:4] * std + mean).clamp(0, 1)
    recon  = (pred_img  * std + mean).clamp(0, 1)

    fig, axes = plt.subplots(3, 4, figsize=(14, 10))
    for i in range(4):
        axes[0, i].imshow(orig[i].permute(1, 2, 0).numpy())
        axes[0, i].set_title('原图')
        axes[0, i].axis('off')

        masked = orig[i].clone()
        m = mask[i].cpu().view(H, W)
        for r in range(H):
            for c in range(W):
                if m[r, c] == 1:
                    masked[:, r*p:(r+1)*p, c*p:(c+1)*p] = 0.5
        axes[1, i].imshow(masked.permute(1, 2, 0).numpy())
        axes[1, i].set_title('掩码输入')
        axes[1, i].axis('off')

        axes[2, i].imshow(recon[i].permute(1, 2, 0).numpy())
        axes[2, i].set_title('重建结果')
        axes[2, i].axis('off')

    plt.suptitle('MAE 重建效果：原图 / 掩码输入 / 重建', y=1.01)
    plt.tight_layout()
    plt.show()


show_reconstruction(mae_model, sample_imgs, cfg, device)

## 10. Fine-tune 分类

In [ ]:
class MAEClassifier(nn.Module):
    def __init__(self, mae, embed_dim, num_classes):
        super().__init__()
        self.mae = mae
        # 在预训练特征上加一个轻量分类头
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        feat = self.mae.encode(x)
        return self.head(feat)


classifier = MAEClassifier(mae_model, cfg.embed_dim, cfg.num_classes).to(device)
finetune_optimizer = optim.Adam(classifier.parameters(), lr=cfg.lr_finetune)
criterion = nn.CrossEntropyLoss()
finetune_history = {'train_acc': [], 'val_acc': []}

for epoch in range(cfg.finetune_epochs):
    classifier.train()
    correct, total = 0, 0
    for images, labels in finetune_loader:
        images, labels = images.to(device), labels.to(device)
        logits = classifier(images)
        loss = criterion(logits, labels)
        finetune_optimizer.zero_grad()
        loss.backward()
        finetune_optimizer.step()
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    train_acc = correct / total

    classifier.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            logits = classifier(images)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)
    val_acc = correct / total

    finetune_history['train_acc'].append(train_acc)
    finetune_history['val_acc'].append(val_acc)
    print(f'Finetune Epoch {epoch+1:2d}/{cfg.finetune_epochs}  train_acc={train_acc:.4f}  val_acc={val_acc:.4f}')